In [ ]:
import numpy as np

ket0 = np.array([1.0 + 0.0j, 0.0 + 0.0j])
I = np.eye(2, dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

CNOT_01 = np.array(
    [
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0],
    ],
    dtype=complex,
)


def kron_all(ops):
    output = ops[0]
    for op in ops[1:]:
        output = np.kron(output, op)
    return output


def ry(theta: float) -> np.ndarray:
    return np.array(
        [
            [np.cos(theta / 2), -np.sin(theta / 2)],
            [np.sin(theta / 2),  np.cos(theta / 2)],
        ],
        dtype=complex,
    )


def rz(theta: float) -> np.ndarray:
    return np.array(
        [
            [np.exp(-0.5j * theta), 0.0],
            [0.0, np.exp(0.5j * theta)],
        ],
        dtype=complex,
    )


def angle_encoding(x: np.ndarray) -> np.ndarray:
    return kron_all([ry(x[0]), ry(x[1])])


def variational_block(params: np.ndarray) -> np.ndarray:
    theta0, theta1, theta2, theta3 = params
    local_z = kron_all([rz(theta0), rz(theta1)])
    local_y = kron_all([ry(theta2), ry(theta3)])
    return CNOT_01 @ local_y @ local_z


def vqc_score(x: np.ndarray, params: np.ndarray) -> float:
    state = kron_all([ket0, ket0])
    state = angle_encoding(x) @ state
    state = variational_block(params) @ state
    observable = kron_all([Z, I])
    return float(np.vdot(state, observable @ state).real)


params = np.array([0.3, -0.4, 0.7, 0.2])
for x in [
    np.array([0.1, 0.2]),
    np.array([1.0, -0.3]),
    np.array([-0.8, 0.6]),
]:
    score = vqc_score(x, params)
    print(x, "score=", score, "class=", int(score >= 0.0))

In [ ]:
import numpy as np

ket0 = np.array([1.0 + 0.0j, 0.0 + 0.0j])
I = np.eye(2, dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
CNOT_01 = np.array(
    [
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0],
    ],
    dtype=complex,
)
rng = np.random.default_rng(7)


def kron_all(ops):
    output = ops[0]
    for op in ops[1:]:
        output = np.kron(output, op)
    return output


def ry(theta: float) -> np.ndarray:
    return np.array(
        [
            [np.cos(theta / 2), -np.sin(theta / 2)],
            [np.sin(theta / 2),  np.cos(theta / 2)],
        ],
        dtype=complex,
    )


def rz(theta: float) -> np.ndarray:
    return np.array(
        [
            [np.exp(-0.5j * theta), 0.0],
            [0.0, np.exp(0.5j * theta)],
        ],
        dtype=complex,
    )


def trainable_layer(params: np.ndarray) -> np.ndarray:
    theta0, theta1, theta2, theta3 = params
    return CNOT_01 @ kron_all([ry(theta0), ry(theta1)]) @ kron_all(
        [rz(theta2), rz(theta3)]
    )


def score(x: np.ndarray, params_by_layer: np.ndarray) -> float:
    state = kron_all([ket0, ket0])
    state = kron_all([ry(x[0]), ry(x[1])]) @ state

    unitary = np.eye(4, dtype=complex)
    for params in params_by_layer:
        unitary = trainable_layer(params) @ unitary

    state = unitary @ state
    observable = kron_all([Z, I])
    return float(np.vdot(state, observable @ state).real)


def mean_abs_gradient(x: np.ndarray, params_by_layer: np.ndarray) -> float:
    flat = params_by_layer.reshape(-1)
    grads = []
    for index in range(len(flat)):
        shifted_plus = flat.copy()
        shifted_minus = flat.copy()
        shifted_plus[index] += np.pi / 2
        shifted_minus[index] -= np.pi / 2

        plus = score(x, shifted_plus.reshape(params_by_layer.shape))
        minus = score(x, shifted_minus.reshape(params_by_layer.shape))
        grads.append(abs(0.5 * (plus - minus)))

    return float(np.mean(grads))


x = np.array([0.4, -0.7])
for depth in [1, 2, 4, 6]:
    sampled = []
    for _ in range(32):
        params = rng.uniform(0.0, 2.0 * np.pi, size=(depth, 4))
        sampled.append(mean_abs_gradient(x, params))

    print(f"depth={depth:>2d} mean|grad|={np.mean(sampled):.6f}")